# RAG with Prompt Caching

This notebook demonstrates a Retrieval-Augmented Generation (RAG) system using OpenAI and LangChain.

## 1. Install Dependencies

Run this cell first to install all required packages.

In [ ]:
!pip install langchain langchain-community langchain-openai pypdf chromadb python-dotenv openai

## 2. Load Environment Variables

Make sure you have a `.env` file in the same directory with your API keys.

In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Verify API key is loaded
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY not found in environment variables. Please check your .env file.")

print("✓ Environment variables loaded successfully")

## 3. Import Required Libraries

In [ ]:
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

## 4. Load and Process Documents

Place your PDF files in the `documents` folder.

In [ ]:
# Load PDFs from documents directory
loader = PyPDFDirectoryLoader("./documents")
docs_raw = loader.load()

print(f"✓ Loaded {len(docs_raw)} document(s)")

# Split documents into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
docs_chunked = splitter.split_documents(docs_raw)

print(f"✓ Split into {len(docs_chunked)} chunks")

## 5. Create Vector Store

Create embeddings and store them in ChromaDB.

In [ ]:
# Create embeddings
embeddings = OpenAIEmbeddings()

# Create vector store
vectorstore = Chroma.from_documents(
    documents=docs_chunked,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

print("✓ Vector store created successfully")

## 6. Set Up Custom Prompt Template

In [ ]:
# Custom prompt template
prompt_template = """
Use the following pieces of context to answer the question at the end.
If you don't know the answer, just say that you don't know, don't try to make up an answer.
Always provide detailed and comprehensive answers based on the context.

Context:
{context}

Question: {question}

Helpful Answer:
"""

PROMPT = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

print("✓ Custom prompt template configured")

## 7. Initialize LLM and RAG Chain

In [ ]:
# Initialize OpenAI LLM
llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0
)

# Create RAG chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore.as_retriever(search_kwargs={"k": 3}),
    chain_type_kwargs={"prompt": PROMPT},
    return_source_documents=True
)

print("✓ RAG chain initialized successfully")

## 8. Query the System

In [ ]:
# Example query
query = "What is the main topic of the documents?"

result = qa_chain.invoke({"query": query})

print("\n" + "="*80)
print("QUESTION:")
print("="*80)
print(query)
print("\n" + "="*80)
print("ANSWER:")
print("="*80)
print(result["result"])
print("\n" + "="*80)
print("SOURCE DOCUMENTS:")
print("="*80)
for i, doc in enumerate(result["source_documents"], 1):
    print(f"\nSource {i}:")
    print(f"Content: {doc.page_content[:200]}...")
    print(f"Metadata: {doc.metadata}")

## 9. Interactive Query Function

In [ ]:
def ask_question(question: str):
    """
    Ask a question to the RAG system and display the answer with sources.
    """
    result = qa_chain.invoke({"query": question})
    
    print("\n" + "="*80)
    print("ANSWER:")
    print("="*80)
    print(result["result"])
    print("\n" + "="*80)
    print(f"Retrieved {len(result['source_documents'])} source document(s)")
    print("="*80)
    
    return result

# Example usage:
# ask_question("Your question here")